# NB8 — Step 4: fine-tune on REAL human errors (Iqra_Extra_IS26)

Steps 2 (more epochs) and 3 (confidence gating) both gave NO F1 gain. The proven
diagnosis: the model over-flags with HIGH confidence -> it doesn't know what REAL
mispronunciations look like (it trained on clean + *synthetic* TTS errors only).

`Iqra_Extra_IS26` = **1,333 real human-mispronounced clips** (cols: audio,
sentence, phoneme_ref=canonical, phoneme_mis=what was ACTUALLY said). The IQRA'26
challenge credited this data as the single most impactful F1 lever.

**Plan:** resume from the 5k model, fine-tune on Iqra_train + Iqra_TTS +
Iqra_Extra_IS26, with the real-error clips **UPWEIGHTED** (repeated) so 1.3k clips
aren't drowned by 126k. Short run (~2-3h). Re-score with NB5.

Gated: accept terms at https://huggingface.co/datasets/IqraEval/Iqra_Extra_IS26 .

In [ ]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" "accelerate==0.34.2" jiwer soundfile librosa
import warnings
warnings.filterwarnings("ignore", message=".*use_reentrant.*")
warnings.filterwarnings("ignore", category=UserWarning)
import torch as _torch
_ol = _torch.load
def _pl(*a, **k):
    k.setdefault("weights_only", False); return _ol(*a, **k)
_torch.load = _pl
print("torch.load patched.")

In [ ]:
import json, os, time
import numpy as np, torch
from dataclasses import dataclass
from datasets import load_dataset, load_from_disk, Audio, concatenate_datasets, Dataset

if torch.cuda.device_count() > 1:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"; print("-> GPU 0 only")
try:
    from kaggle_secrets import UserSecretsClient
    _t = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _t; os.environ["HUGGING_FACE_HUB_TOKEN"] = _t
    print("HF_TOKEN loaded.")
except Exception as e:
    print("No HF_TOKEN (need it for gated Iqra_Extra_IS26):", e)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")

MODEL_NAME = "facebook/w2v-bert-2.0"
SAMPLING_RATE = 16_000
PHONEME_COL = "phoneme_ref"

# Cached Iqra_train + Iqra_TTS (your iqra-cache dataset).
CACHE_ROOT = "/kaggle/input/datasets/qasimmaajid/iqra-cache/iqra_prep"
LOCAL_TRAIN_DISK = f"{CACHE_ROOT}/iqra_train_arrow"
LOCAL_TTS_DISK   = f"{CACHE_ROOT}/iqra_tts_arrow"
VOCAB_PATH       = f"{CACHE_ROOT}/phoneme_vocab.json"

# Resume from the BEST model (5k). EDIT to your saved checkpoint path.
RESUME_CKPT = "/kaggle/input/datasets/qasimmaajid/quran-phoneme-model/w2vbert-quran-phoneme/checkpoint-5000"
OUTPUT_DIR  = "/kaggle/working/w2vbert-quran-phoneme-real"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXTRA_REPEAT = 20     # repeat the 1.3k real-error clips ~20x (~26k) so they matter
MAX_STEPS    = 8000   # ~+3000 from 5k, short (~2.5h), defensive save below
EVAL_STEPS   = 1000
print("resume exists:", os.path.isdir(RESUME_CKPT), "| vocab:", os.path.exists(VOCAB_PATH))

In [ ]:
from transformers import AutoFeatureExtractor, Wav2Vec2CTCTokenizer
with open(VOCAB_PATH, encoding="utf-8") as f:
    vocab = json.load(f)
with open(os.path.join(OUTPUT_DIR, "vocab.json"), "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)
tokenizer = Wav2Vec2CTCTokenizer(os.path.join(OUTPUT_DIR, "vocab.json"),
                                 unk_token="<unk>", pad_token="<pad>", word_delimiter_token=" ")
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
print("vocab size:", len(vocab))

In [ ]:
# ---- Build training mix -----------------------------------------------------
# Iqra_train -> phoneme_ref (clean) ; Iqra_TTS + Iqra_Extra_IS26 -> phoneme_mis
# (what was actually SAID). The real-error set is UPWEIGHTED via repetition.
def normalise(ds, label_col):
    keep = {"audio", label_col}
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep])
    if label_col != "labels_str": ds = ds.rename_column(label_col, "labels_str")
    return ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

full = load_from_disk(LOCAL_TRAIN_DISK); train, dev = full["train"], full["dev"]
train = normalise(train, PHONEME_COL); dev = normalise(dev, PHONEME_COL)

tf = load_from_disk(LOCAL_TTS_DISK)
tts = tf["train"] if "train" in getattr(tf, "keys", lambda: [])() else tf
tts = normalise(tts, "phoneme_mis" if "phoneme_mis" in tts.column_names else PHONEME_COL)

# Real-error data (gated download). Label = phoneme_mis (what was said).
extra = load_dataset("IqraEval/Iqra_Extra_IS26", split="train")
print("Iqra_Extra_IS26 columns:", extra.column_names, "| rows:", len(extra))
extra = normalise(extra, "phoneme_mis" if "phoneme_mis" in extra.column_names else PHONEME_COL)

# Upweight real errors by repeating the index list.
extra_up = concatenate_datasets([extra] * EXTRA_REPEAT)
print(f"real-error upweighted: {len(extra)} -> {len(extra_up)} (x{EXTRA_REPEAT})")

train = concatenate_datasets([train, tts, extra_up])
train = train.shuffle(seed=0, keep_in_memory=True)
print("TOTAL train:", len(train), "| dev:", len(dev))

In [ ]:
unk_id = tokenizer.convert_tokens_to_ids("<unk>")
def encode_labels(s): return [vocab.get(t, unk_id) for t in str(s).split() if t]
def make_transform(b):
    return {"input_features": [feature_extractor(a["array"], sampling_rate=SAMPLING_RATE).input_features[0] for a in b["audio"]],
            "labels": [encode_labels(s) for s in b["labels_str"]]}
train_proc = train.with_transform(make_transform)
dev_proc   = dev.with_transform(make_transform)

@dataclass
class CTCCollator:
    feature_extractor: object
    tokenizer: object
    def __call__(self, features):
        batch = self.feature_extractor.pad([{"input_features": f["input_features"]} for f in features], return_tensors="pt")
        labs = self.tokenizer.pad([{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        batch["labels"] = labs["input_ids"].masked_fill(labs.attention_mask.ne(1), -100)
        return batch
collator = CTCCollator(feature_extractor, tokenizer)

import jiwer
id2tok = {v: k for k, v in vocab.items()}
def decode_ids(ids):
    out, prev, blank = [], None, vocab["<pad>"]
    for i in ids:
        if i != prev and i != blank: out.append(id2tok.get(int(i), "<unk>"))
        prev = i
    return " ".join(out)
def compute_metrics(pred):
    pi = np.argmax(pred.predictions, axis=-1)
    lab = np.where(pred.label_ids == -100, vocab["<pad>"], pred.label_ids)
    return {"per": jiwer.wer([decode_ids(l) for l in lab], [decode_ids(p) for p in pi])}
print("transforms + metrics ready")

In [ ]:
from transformers import Wav2Vec2BertForCTC, TrainingArguments, Trainer, EarlyStoppingCallback
model = Wav2Vec2BertForCTC.from_pretrained(
    RESUME_CKPT, vocab_size=len(vocab), pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean", add_adapter=True)
model.config.ctc_zero_infinity = True

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8, per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-5, lr_scheduler_type="cosine", warmup_steps=200,
    max_steps=MAX_STEPS,
    fp16=True, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="steps", eval_steps=EVAL_STEPS, save_steps=EVAL_STEPS,
    save_total_limit=1, save_safetensors=True,
    logging_steps=500, disable_tqdm=True,
    load_best_model_at_end=True, metric_for_best_model="per", greater_is_better=False,
    dataloader_num_workers=2,
    remove_unused_columns=False, label_names=["labels"], report_to="none",
)
# NOTE: we do NOT resume optimizer state here (fresh fine-tune on the new mix),
# we just load the 5k WEIGHTS as the starting point. So no resume_from_checkpoint.
trainer = Trainer(model=model, args=args, train_dataset=train_proc, eval_dataset=dev_proc,
                  data_collator=collator, compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
try: trainer.tokenizer = tokenizer
except Exception: pass
print("trainer ready")

In [ ]:
trainer.train()
try:
    trainer.save_model(os.path.join(OUTPUT_DIR, "final"))
    tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "final"))
    feature_extractor.save_pretrained(os.path.join(OUTPUT_DIR, "final"))
    print("saved final ->", os.path.join(OUTPUT_DIR, "final"))
except Exception as e:
    print("final save failed (disk?):", e)
print("survivors:", [p for p in os.listdir(OUTPUT_DIR) if p.startswith("checkpoint") or p == "final"])

### After this run
1. Commit -> Create Dataset from `final` (or the last `checkpoint-XXXX`).
2. Re-run **NB5** with `MODEL_DIR` -> the new official F1.
3. If F1 rose past 0.4618 -> real-error data worked (expected; it's the proven lever).
   If flat -> tune `EXTRA_REPEAT` (try 10 or 40) or move to Step 5 (LM rescoring).

**Why no optimizer resume:** we're fine-tuning the 5k WEIGHTS on a NEW data mix
(now including real errors). A fresh optimizer + short cosine schedule lets the
model adapt to the real-error distribution without being anchored to the old run.